In [ ]:
import os
# Parameters — update before running interactively
ci_target = "ephemeral_ci"
prod_state_path = "abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/Files/ci-artifacts/prod-state"
project_zip_abfss = "abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/Files/ci-artifacts/dbt-project/HEAD_SHA/project.zip"
lakehouse_name = "LAKEHOUSE_NAME"
lakehouse_id = "LAKEHOUSE_ID"
workspace_id = "WORKSPACE_ID"
workspace_name = "WORKSPACE_NAME"
schema_name = "dbo"
head_sha = "HEAD_SHA"


In [ ]:
# Re-hydrate dbt project from OneLake zip (per-session — /tmp is session-scoped)
import os
import zipfile
import urllib.request

_https_url = project_zip_abfss.replace('abfss://', 'https://onelake.dfs.fabric.microsoft.com/', 1)
_https_url = _https_url.replace('@onelake.dfs.fabric.microsoft.com', '', 1)
_token = notebookutils.credentials.getToken('storage')  # noqa: F821
_req = urllib.request.Request(_https_url, headers={'Authorization': f'Bearer {_token}'})
os.makedirs('/tmp/dbt_project', exist_ok=True)
with urllib.request.urlopen(_req) as _resp:
    with open('/tmp/project.zip', 'wb') as _f:
        _f.write(_resp.read())
with zipfile.ZipFile('/tmp/project.zip', 'r') as _zf:
    _zf.extractall('/tmp/dbt_project')
print('dbt project re-hydrated to /tmp/dbt_project', flush=True)


In [ ]:
# Download prod-state manifest from OneLake (ABFSS → local path)
# No-op when prod_state_path is already a local path (e.g. greenfield ./prod-state).
if prod_state_path.startswith('abfss://'):
    import os
    import urllib.request
    https_url = prod_state_path.replace('abfss://', 'https://onelake.dfs.fabric.microsoft.com/', 1)
    https_url = https_url.replace('@onelake.dfs.fabric.microsoft.com', '', 1)
    manifest_url = https_url.rstrip('/') + '/manifest.json'
    local_dir = '/tmp/prod-state'
    os.makedirs(local_dir, exist_ok=True)
    token = notebookutils.credentials.getToken('storage')  # noqa: F821
    req = urllib.request.Request(manifest_url, headers={'Authorization': f'Bearer {token}'})
    with urllib.request.urlopen(req) as resp:
        with open(f'{local_dir}/manifest.json', 'wb') as f:
            f.write(resp.read())
    local_prod_state_path = local_dir
else:
    local_prod_state_path = prod_state_path

In [ ]:
# Command assembly
_deps  = "dbt deps"
_clone = f"dbt clone --select state:modified+ --exclude config.materialized:view --defer --state {local_prod_state_path} --target-path target/clone --profiles-dir /tmp/dbt_profiles --target {ci_target}"
run = f"dbt run --select state:modified+ --defer --state {local_prod_state_path} --target-path target/build --profiles-dir /tmp/dbt_profiles --target {ci_target}"
# Build unit test command from deployment manifest — indirect selection scopes to state:modified+ models.
# dbt unit test nodes are not reachable via state:modified+ graph traversal;
# naming models directly triggers indirect selection which includes their unit tests.
import json as _json  # noqa: E402
_manifest_path = f"/lakehouse/default/Files/ci-artifacts/deployment-manifest-{head_sha}.json"  # noqa: F821
_manifest_models = [
    a["name"].split(".")[-1]
    for a in _json.load(open(_manifest_path)).get("artifacts", [])
    if a.get("materialized") != "ephemeral"
]
_unit = (
    f"dbt test --select {' '.join(_manifest_models)},test_type:unit"
    f" --target-path target/unit --profiles-dir /tmp/dbt_profiles --target {ci_target}"
    if _manifest_models else
    f"dbt test --select test_type:unit"
    f" --target-path target/unit --profiles-dir /tmp/dbt_profiles --target {ci_target}"
)
_data  = f"dbt test --select state:modified+ --exclude test_type:unit --store-failures --defer --state {local_prod_state_path} --target-path target/data --profiles-dir /tmp/dbt_profiles --target {ci_target}"

clone_command     = [_deps, _clone]
build_command     = [_deps, run]
# Per ADR 0002 / VD-2175: run the build cell before this cell — gate-3 unit tests now depend on Gate 2's output, matching CI order.
unit_test_command = [_deps, _unit]
data_test_command = [_deps, _data]


In [ ]:
!pip install dbt-fabricspark -q

In [ ]:
# Write authentication: fabric_notebook dbt profile for this session (AC-32, AC-34)
import os
import yaml

_profile_dir = '/tmp/dbt_profiles'
os.makedirs(_profile_dir, exist_ok=True)
with open('/tmp/dbt_project/dbt_project.yml') as _dbt_proj_f:
    _profile_name = yaml.safe_load(_dbt_proj_f)['profile']
with open(f'{_profile_dir}/profiles.yml', 'w') as _pf:
    yaml.dump(
        {
            _profile_name: {
                'target': ci_target,
                'outputs': {
                    ci_target: {
                        'type': 'fabricspark',
                        'method': 'livy',
                        'authentication': 'fabric_notebook',
                        'workspaceid': workspace_id,
                        'lakehouse': lakehouse_name,
                        'schema': schema_name,
                        'connect_retries': 3,
                        'connect_timeout': 300,
                    }
                },
            }
        },
        _pf,
    )
print(f'Wrote fabric_notebook profile to {_profile_dir}/profiles.yml', flush=True)


In [ ]:
# Clone (interactive) — AC-32: dbt CLI with authentication: fabric_notebook
import subprocess
for _cmd in clone_command:
    _r = subprocess.run(_cmd, shell=True, cwd='/tmp/dbt_project')
    if _r.returncode != 0:
        raise RuntimeError(f'dbt command failed (exit {_r.returncode}): {_cmd}')


In [ ]:
# Build (interactive) — AC-32: dbt CLI with authentication: fabric_notebook
import subprocess
for _cmd in build_command:
    _r = subprocess.run(_cmd, shell=True, cwd='/tmp/dbt_project')
    if _r.returncode != 0:
        raise RuntimeError(f'dbt command failed (exit {_r.returncode}): {_cmd}')


In [ ]:
# Unit Test (interactive) — AC-32: dbt CLI with authentication: fabric_notebook
import subprocess
for _cmd in unit_test_command:
    _r = subprocess.run(_cmd, shell=True, cwd='/tmp/dbt_project')
    if _r.returncode != 0:
        raise RuntimeError(f'dbt command failed (exit {_r.returncode}): {_cmd}')


In [ ]:
# Data Test (interactive) — AC-32: dbt CLI with authentication: fabric_notebook
import subprocess
for _cmd in data_test_command:
    _r = subprocess.run(_cmd, shell=True, cwd='/tmp/dbt_project')
    if _r.returncode != 0:
        raise RuntimeError(f'dbt command failed (exit {_r.returncode}): {_cmd}')
